In [7]:
import ee
import multiprocessing
from tqdm import tqdm
from retry import retry
import requests
import numpy as np

In [2]:
ee.Authenticate()
ee.Initialize(project='ee-kerrilu')
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

In [3]:
# https://gorelick.medium.com/fast-er-downloads-a2abd512aa26
NAIP = ee.ImageCollection("USDA/NAIP/DOQQ")
CDL = ee.ImageCollection("USDA/NASS/CDL")
grid_buffered_US = ee.FeatureCollection("projects/ee-kerrilu/assets/US-fields/grid_buffered_US_final")

RGB = NAIP.filterDate('2021-01-01', '2024-01-01')
CDL = CDL.filter(ee.Filter.date('2023-01-01', '2023-12-31')).first().select('cultivated').gt(1)

@retry(tries=10, delay=1, backoff=2)
def getResult(enumerated_tile):
    i, tile = enumerated_tile
    tile = ee.Geometry.Polygon(tile['coordinates'])
    images = RGB.filterBounds(tile).sort('system:time_start')
    image_mosaic = images.mosaic()
    image = image_mosaic.clip(tile)
    image_CDL = CDL.clip(tile)

    try:
        filename = f'inference_data/images/{(i+1):05d}.tif'
        url = image.getDownloadURL({
            'dimensions':'2048x2048',
            'crs':'EPSG:5070',
            'format': 'GEO_TIFF'
        })
        r = requests.get(url, stream=True)
        if r.status_code != 200:
            raise r.raise_for_status()
        with open(filename, 'wb') as out_file:
            out_file.write(r.content)
    
        filename = f'inference_data/CDL/{(i+1):05d}.tif'
        url = image_CDL.getDownloadURL({
            'dimensions':'2048x2048',
            'crs':'EPSG:5070',
            'format': 'GEO_TIFF'
        })
        r = requests.get(url, stream=True)
        if r.status_code != 200:
            raise r.raise_for_status()
        with open(filename, 'wb') as out_file:
            out_file.write(r.content)
    
    except Exception as e:
        print(f"{e}\n Tile number: {i}\n")

In [4]:
# DOWNLOAD IMAGES 
tiles = grid_buffered_US.aggregate_array('.geo').getInfo()[:18000]
with multiprocessing.Pool(25) as pool:
    results = list(tqdm(pool.imap(getResult, enumerate(tiles)), total=len(tiles)))
    pool.close()
    pool.join()

100%|██████████| 18000/18000 [4:30:09<00:00,  1.11it/s]  


In [4]:
# DOWNLOAD IMAGES 
tiles = grid_buffered_US.aggregate_array('.geo').getInfo()[18000:36000]
with multiprocessing.Pool(25) as pool:
    results = list(tqdm(pool.imap(getResult, enumerate(tiles, start=18000)), total=len(tiles)))
    pool.close()
    pool.join()

 26%|██▌       | 4620/18000 [49:51<1:34:31,  2.36it/s] 

429 Client Error: Too Many Requests for url: https://earthengine-highvolume.googleapis.com/v1/projects/718560042697/thumbnails/6d81c1f9c5e1ee1907a31fbcb3ffe078-ecd95deb7b383f9be8aa2aa178dd125c:getPixels
 Tile number: 22640

429 Client Error: Too Many Requests for url: https://earthengine-highvolume.googleapis.com/v1/projects/718560042697/thumbnails/069bdd3250f635c8ad4c425ee297740c-a51f6fb66f3109b31c5933276988d131:getPixels
 Tile number: 22646

429 Client Error: Too Many Requests for url: https://earthengine-highvolume.googleapis.com/v1/projects/718560042697/thumbnails/b220e5074bfa17114e48ff969f5d2025-f8af7d9ea50d21651b3b0a132be45941:getPixels
 Tile number: 22647



 27%|██▋       | 4896/18000 [53:06<3:51:17,  1.06s/it] 

429 Client Error: Too Many Requests for url: https://earthengine-highvolume.googleapis.com/v1/projects/718560042697/thumbnails/7707ebbe1d730f1f2f5febbc25a9998e-cecc1113541bf8938a5d488bed7f6294:getPixels
 Tile number: 22950



 28%|██▊       | 5036/18000 [54:29<2:54:12,  1.24it/s]

...

 84%|████████▍ | 15210/18000 [2:49:03<28:21,  1.64it/s]  

503 Server Error: Service Unavailable for url: https://earthengine-highvolume.googleapis.com/v1/projects/718560042697/thumbnails/91c77b5296f73c6c4c0b48f938003fd4-5f3413a5bf9d0707c77ef860c6543b12:getPixels
 Tile number: 33252



 90%|█████████ | 16217/18000 [3:01:56<15:07,  1.96it/s]  

429 Client Error: Too Many Requests for url: https://earthengine-highvolume.googleapis.com/v1/projects/718560042697/thumbnails/d726c76dc04f2cbfa4c8d0ad73e2f36b-5bc0e27e285026782bbaea5bfcac7df4:getPixels
 Tile number: 34252



100%|██████████| 18000/18000 [3:24:56<00:00,  1.46it/s]  


In [4]:
# DOWNLOAD IMAGES 
tiles = grid_buffered_US.aggregate_array('.geo').getInfo()[36000:]
with multiprocessing.Pool(25) as pool:
    results = list(tqdm(pool.imap(getResult, enumerate(tiles, start=36000)), total=len(tiles)))
    pool.close()
    pool.join()

 16%|█▋        | 2855/17379 [40:16<3:15:12,  1.24it/s] 

429 Client Error: Too Many Requests for url: https://earthengine-highvolume.googleapis.com/v1/projects/718560042697/thumbnails/72d40795575049080cd26f12a3a0fffa-75b13b61decd57624435d27b24417679:getPixels
 Tile number: 38882

429 Client Error: Too Many Requests for url: https://earthengine-highvolume.googleapis.com/v1/projects/718560042697/thumbnails/20f5c5db1f6b6cfb6561a151ad96e34e-72e96b6f246e07643d9a95906503163b:getPixels
 Tile number: 38870



 45%|████▍     | 7784/17379 [1:40:08<58:05,  2.75it/s]   

429 Client Error: Too Many Requests for url: https://earthengine-highvolume.googleapis.com/v1/projects/718560042697/thumbnails/8e7c05ef7b58406e1efc8f9367322807-db6dee9bcc6b0845ddb34d5bebc3168a:getPixels
 Tile number: 43792



 49%|████▉     | 8483/17379 [1:46:36<1:00:04,  2.47it/s]

...

 98%|█████████▊| 17117/17379 [3:17:27<03:26,  1.27it/s]

429 Client Error: Too Many Requests for url: https://earthengine-highvolume.googleapis.com/v1/projects/718560042697/thumbnails/8688d9cfecb50ea0a5b33947bab0d24d-17bc5a657953968f368270712246136c:getPixels
 Tile number: 53153



 99%|█████████▉| 17191/17379 [3:18:41<03:23,  1.08s/it]

429 Client Error: Too Many Requests for url: https://earthengine-highvolume.googleapis.com/v1/projects/718560042697/thumbnails/5f21c09217e2235e5bd2ca636f45bf8e-74a281fe7234129d072131e1ac719632:getPixels
 Tile number: 53265



100%|██████████| 17379/17379 [3:21:17<00:00,  1.44it/s]


# Fix missing tiles (due to "too many requests for url" or "service unavailable for url" errors)

In [8]:
missing_tile_indices = [22640, 22646, 22647, 22950, 23056, 23077, 27526, 27530, 33252, 34252, 38882, 38870, 43792, 44518, 48326, 48615, 49217, 53068, 53105, 53114, 53147, 53150, 53149, 53153, 53265]
tiles = np.array(grid_buffered_US.aggregate_array('.geo').getInfo())[missing_tile_indices]
with multiprocessing.Pool(25) as pool:
    results = list(tqdm(pool.imap(getResult, zip(missing_tile_indices, tiles)), total=len(tiles)))
    pool.close()
    pool.join()

100%|██████████| 25/25 [01:48<00:00,  4.33s/it]
